<a href="https://colab.research.google.com/github/amberjill-li/Machine-Learning-Assignment/blob/main/COMP3010_Assignment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **COMP3010 - Machine Learning**

**Amberjill Li Kwong Ken**

**Kaggle Team Name: 21469970**

**Final Private Leaderboard MAPE: 0.1926**


#**Data Preprocessing**

Effective data preprocessing is crucial for successful machine learning applications. This stage
includes selecting relevant features, removing redundant or irrelevant ones, and potentially
creating new features. You are also expected to adapt data formats and types based on
the requirements of your chosen model. A particular emphasis should be placed on data
cleaning, which may include the following:

In [ ]:
# Imports
import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer


## **Identifying and Handling Missing Values:**
Examine the dataset for missing or
incomplete entries. Choose appropriate strategies such as imputation or deletion, depending on the context and potential impact on model performance.


In [ ]:
# Mount Google Drive & Load Data
from google.colab import drive
drive.mount('/content/drive')

# Then point to wherever you saved the files in your Drive
train = pd.read_csv('/content/drive/MyDrive/COMP3010/train.csv')
test  = pd.read_csv('/content/drive/MyDrive/COMP3010/test.csv')

print(f"Train shape: {train.shape}")   # (10050, 25)
print(f"Test shape:  {test.shape}")    # (3203, 24)
print()
print("Columns:", list(train.columns))


Mounted at /content/drive
Train shape: (10050, 25)
Test shape:  (3203, 24)

Columns: ['ID', 'Tank Failure Pressure (bar)', 'Liquid Ratio (%)', 'Tank Width (m)', 'Tank Length (m)', 'Tank Height (m)', 'BLEVE Height (m)', 'Vapour Height (m)', 'Vapour Temperature (K)', 'Liquid Temperature (K)', 'Obstacle Distance to BLEVE (m)', 'Obstacle Width (m)', 'Obstacle Height (m)', 'Obstacle Thickness (m)', 'Obstacle Angle', 'Status', 'Liquid Critical Pressure (bar)', 'Liquid Boiling Temperature (K)', 'Liquid Critical Temperature (K)', 'Sensor ID', 'Sensor Position Side', 'Sensor Position x', 'Sensor Position y', 'Sensor Position z', 'Target Pressure (bar)']


In [ ]:
# Missing Value Summary

missing_count = train.isnull().sum()
missing_pct   = (missing_count / len(train) * 100).round(2)

mv_summary = pd.DataFrame({
    'Missing Count': missing_count,
    'Missing %':     missing_pct
})
mv_summary = mv_summary[mv_summary['Missing Count'] > 0]
print(mv_summary)

                                 Missing Count  Missing %
ID                                           5       0.05
Tank Failure Pressure (bar)                  7       0.07
Liquid Ratio (%)                             9       0.09
Tank Width (m)                               9       0.09
Tank Length (m)                              9       0.09
Tank Height (m)                              8       0.08
BLEVE Height (m)                            10       0.10
Vapour Height (m)                            9       0.09
Vapour Temperature (K)                      28       0.28
Liquid Temperature (K)                      27       0.27
Obstacle Distance to BLEVE (m)               8       0.08
Obstacle Width (m)                           6       0.06
Obstacle Height (m)                          6       0.06
Obstacle Thickness (m)                       7       0.07
Obstacle Angle                               8       0.08
Status                                       7       0.07
Liquid Critica

In [ ]:
#Identify rows missing many columns

n_missing_per_row = train.isnull().sum(axis=1)
bad_rows = train[n_missing_per_row > 5]
print(f"\nRows with >5 missing columns: {len(bad_rows)}")
print(f"These rows also have missing target: "
      f"{bad_rows['Target Pressure (bar)'].isnull().sum()} / {len(bad_rows)}")

#Dropping these 10 bad rows
train = train[n_missing_per_row <= 5].reset_index(drop=True)
print(f"\nTrain shape after dropping bad rows: {train.shape}")  # (10040, 25)


Rows with >5 missing columns: 10
These rows also have missing target: 10 / 10

Train shape after dropping bad rows: (10040, 25)


In [ ]:
# Drop the ID column
train = train.drop(columns=['ID'])
test  = test.drop(columns=['ID'])

In [ ]:
# Impute remaining missing values
# Separate feature types
numeric_cols     = train.select_dtypes(include='number').columns.tolist()
numeric_cols.remove('Target Pressure (bar)')   # don't impute the target

categorical_cols = ['Status']

# Numeric imputer
num_imputer = SimpleImputer(strategy='median')
train[numeric_cols] = num_imputer.fit_transform(train[numeric_cols])
test[numeric_cols]  = num_imputer.transform(test[numeric_cols])

# Categorical imputer
cat_imputer = SimpleImputer(strategy='most_frequent')
train[categorical_cols] = cat_imputer.fit_transform(train[categorical_cols])
test[categorical_cols]  = cat_imputer.transform(test[categorical_cols])

In [ ]:
print("Remaining missing in train:", train.isnull().sum().sum())  # 0
print("Remaining missing in test: ", test.isnull().sum().sum())   # 0

Remaining missing in train: 0
Remaining missing in test:  0


##**Outlier Detection and Treatment:**
Detect and address any outliers that may distort your analysis. Apply suitable statistical techniques to correct or exclude these
anomalies

In [ ]:
#IQR Outlier Scan
numeric_cols = train.select_dtypes(include='number').columns.tolist()
numeric_cols.remove('Target Pressure (bar)')  # handle target separately

print("=== IQR Outlier Summary (features only) ===")
for col in numeric_cols:
    Q1  = train[col].quantile(0.25)
    Q3  = train[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    n_out = ((train[col] < lower) | (train[col] > upper)).sum()
    if n_out > 0:
        print(f"  {col}: {n_out} outliers | "
              f"range [{train[col].min():.2f}, {train[col].max():.2f}] | "
              f"IQR bounds [{lower:.2f}, {upper:.2f}]")

=== IQR Outlier Summary (features only) ===
  Tank Failure Pressure (bar): 50 outliers | range [4.92, 4995.62] | IQR bounds [-13.97, 59.88]
  Vapour Height (m): 71 outliers | range [0.18, 2.62] | IQR bounds [-0.80, 2.40]
  Sensor Position y: 10 outliers | range [-9.05, 19.55] | IQR bounds [-10.10, 16.30]
  Sensor Position z: 92 outliers | range [-2.80, 16.70] | IQR bounds [-9.45, 14.15]


In [ ]:
# Tank Failure Pressure — cap corrupted values
cap_tfp = train['Tank Failure Pressure (bar)'].quantile(0.99)
print(f"\nCapping 'Tank Failure Pressure' at 99th pct: {cap_tfp:.2f} bar")

train['Tank Failure Pressure (bar)'] = train['Tank Failure Pressure (bar)'].clip(upper=cap_tfp)
test['Tank Failure Pressure (bar)']  = test['Tank Failure Pressure (bar)'].clip(upper=cap_tfp)

print(f"New max in train: {train['Tank Failure Pressure (bar)'].max():.2f} bar")



Capping 'Tank Failure Pressure' at 99th pct: 41.87 bar
New max in train: 41.87 bar


In [ ]:
# Sensor Position columns — keep as-is

print("\nSensor Position z unique value count:", train['Sensor Position z'].nunique())
print("These are valid discrete sensor coordinates — no treatment needed.")



Sensor Position z unique value count: 180
These are valid discrete sensor coordinates — no treatment needed.


In [ ]:
#Vapour Height

print(f"\nVapour Height max: {train['Vapour Height (m)'].max():.3f} m — physically plausible, no treatment.")


Vapour Height max: 2.620 m — physically plausible, no treatment.


In [ ]:
#Target Pressure
print("\n=== Target Pressure: before log-transform ===")
print(train['Target Pressure (bar)'].describe().round(4))

train['Log Target Pressure'] = np.log(train['Target Pressure (bar)'])

print("\n=== Log Target Pressure: after transform ===")
print(train['Log Target Pressure'].describe().round(4))

# --- Cell 13: Verify final state ---
print(f"\nFinal train shape: {train.shape}")
print(f"Final test shape:  {test.shape}")
print("Any remaining outlier concerns: Tank Failure Pressure capped, all others retained.")


=== Target Pressure: before log-transform ===
count    10040.0000
mean         0.3605
std          0.4953
min          0.0161
25%          0.1022
50%          0.2063
75%          0.4132
max          9.1705
Name: Target Pressure (bar), dtype: float64

=== Log Target Pressure: after transform ===
count    10040.0000
mean        -1.5443
std          0.9907
min         -4.1290
25%         -2.2807
50%         -1.5782
75%         -0.8839
max          2.2160
Name: Log Target Pressure, dtype: float64

Final train shape: (10040, 25)
Final test shape:  (3203, 23)
Any remaining outlier concerns: Tank Failure Pressure capped, all others retained.


## **Duplicate Removal:**
Ensure dataset integrity by identifying and removing duplicate
records, which can otherwise bias the model.


In [ ]:
#Identifying Duplicates

n_dupes = train.duplicated().sum()
print(f"Duplicate rows found: {n_dupes}")  # 50

dupe_indices = train[train.duplicated()].index.tolist()
print(f"Duplicate row indices: {min(dupe_indices)} to {max(dupe_indices)}")

Duplicate rows found: 50
Duplicate row indices: 9990 to 10039


In [ ]:
#Remove Duplicates
train = train.drop_duplicates(keep='first').reset_index(drop=True)

print(f"\nTrain shape after duplicate removal: {train.shape}")  # (9990, 25)
print(f"Duplicates remaining: {train.duplicated().sum()}")      # 0

# No duplicates exist in test — no action needed there.
print(f"Duplicates in test:   {test.duplicated().sum()}")       # 0


Train shape after duplicate removal: (9990, 25)
Duplicates remaining: 0
Duplicates in test:   0


## **Correcting Inaccurate Entries:**

Carefully inspect the data for incorrect values and
rectify them to maintain dataset quality.


In [ ]:
# Correcting Inaccurate Entries — Standardise Status column
# The Status column contains 9 inconsistent string variants of 2 categories

print("Unique Status values before cleaning:")
print(train['Status'].unique())

status_map = {
    'Superheated': 'Superheated', 'superheated': 'Superheated',
    'Superheat':   'Superheated', 'Saperheated': 'Superheated',
    'Subcooled':   'Subcooled',   'subcooled':   'Subcooled',
    'Subcool':     'Subcooled',   'Subcoled':    'Subcooled',
}

train['Status'] = train['Status'].map(status_map).fillna('Subcooled')
test['Status']  = test['Status'].map(status_map).fillna('Subcooled')

print("\nUnique Status values after cleaning:")
print(train['Status'].unique())  # should be exactly ['Superheated', 'Subcooled']

Unique Status values before cleaning:
['Superheated' 'Subcooled' 'Subcool' 'subcooled' 'superheated' 'Subcoled'
 'Superheat' 'Saperheated']

Unique Status values after cleaning:
['Superheated' 'Subcooled']


## **Feature Selection:**
Identify and retain only the features that exhibit meaningful correlation with the target variable. Consider building a “sparse” model using a reduced
set of the most informative features

In [ ]:
# Pearson Correlation with Target
target = 'Target Pressure (bar)'
numeric_cols = train.select_dtypes(include='number').columns.tolist()
numeric_cols = [c for c in numeric_cols if c != target]

corr = train[numeric_cols + [target]].corr()[target].drop(target)
corr_sorted = corr.abs().sort_values(ascending=False)

print("=== Pearson Correlation with Target (absolute) ===")
for col, val in corr_sorted.items():
    print(f"  {val:.4f}  {col}")

=== Pearson Correlation with Target (absolute) ===
  0.7840  Log Target Pressure
  0.2808  Sensor Position x
  0.2722  Vapour Height (m)
  0.2057  Obstacle Distance to BLEVE (m)
  0.1970  Tank Height (m)
  0.1953  Sensor Position z
  0.1895  Sensor Position y
  0.1643  Tank Length (m)
  0.1388  Liquid Ratio (%)
  0.1137  Obstacle Height (m)
  0.0689  Obstacle Width (m)
  0.0649  Tank Failure Pressure (bar)
  0.0647  Liquid Temperature (K)
  0.0645  Sensor ID
  0.0602  BLEVE Height (m)
  0.0513  Tank Width (m)
  0.0451  Liquid Boiling Temperature (K)
  0.0451  Liquid Critical Temperature (K)
  0.0446  Liquid Critical Pressure (bar)
  0.0314  Obstacle Angle
  0.0151  Obstacle Thickness (m)
  0.0058  Sensor Position Side
  0.0033  Vapour Temperature (K)


In [ ]:
#Random Forest Feature Importances
# Note: Status typos were already standardised in the Correcting Inaccurate
# Entries step above. We encode it to numeric here for the RF selector.
train['Status'] = (train['Status'] == 'Superheated').astype(int)
test['Status']  = (test['Status'] == 'Superheated').astype(int)

from sklearn.ensemble import RandomForestRegressor

X = train.drop(columns=[target, 'Log Target Pressure'])
y = train['Log Target Pressure']

rf_selector = RandomForestRegressor(n_estimators=100, random_state=0, n_jobs=-1)
rf_selector.fit(X, y)

importances = pd.Series(rf_selector.feature_importances_, index=X.columns)
importances = importances.sort_values(ascending=False)

print("\n=== Random Forest Feature Importances ===")
for col, val in importances.items():
    marker = '  <-- near zero' if val < 0.01 else ''
    print(f"  {val:.4f}  {col}{marker}")


=== Random Forest Feature Importances ===
  0.3211  Sensor Position Side
  0.2725  Sensor ID
  0.0795  Vapour Height (m)
  0.0643  Tank Length (m)
  0.0547  Sensor Position y
  0.0316  Tank Failure Pressure (bar)
  0.0290  Sensor Position z
  0.0275  Sensor Position x
  0.0227  Tank Width (m)
  0.0166  Obstacle Distance to BLEVE (m)
  0.0111  Tank Height (m)
  0.0107  Obstacle Width (m)
  0.0106  Obstacle Height (m)
  0.0099  Liquid Temperature (K)  <-- near zero
  0.0092  Obstacle Thickness (m)  <-- near zero
  0.0078  Vapour Temperature (K)  <-- near zero
  0.0072  BLEVE Height (m)  <-- near zero
  0.0066  Obstacle Angle  <-- near zero
  0.0053  Liquid Ratio (%)  <-- near zero
  0.0010  Status  <-- near zero
  0.0003  Liquid Boiling Temperature (K)  <-- near zero
  0.0003  Liquid Critical Pressure (bar)  <-- near zero
  0.0003  Liquid Critical Temperature (K)  <-- near zero


In [ ]:
#Cross-validate full vs reduced feature set
from sklearn.model_selection import cross_val_score

drop_cols = [
    'Status',
    'Liquid Critical Pressure (bar)',
    'Liquid Boiling Temperature (K)',
    'Liquid Critical Temperature (K)',
]

X_full    = train.drop(columns=[target, 'Log Target Pressure'])
X_reduced = X_full.drop(columns=drop_cols)
y         = train['Log Target Pressure']

rf = RandomForestRegressor(n_estimators=100, random_state=0, n_jobs=-1)

r2_full    = cross_val_score(rf, X_full,    y, cv=5, scoring='r2').mean()
r2_reduced = cross_val_score(rf, X_reduced, y, cv=5, scoring='r2').mean()

print(f"\nR² with all features:          {r2_full:.4f}")
print(f"R² without 4 weakest features: {r2_reduced:.4f}")
print(f"Difference:                    {r2_reduced - r2_full:.4f}")


R² with all features:          0.8757
R² without 4 weakest features: 0.8751
Difference:                    -0.0006


In [ ]:
#Drop the 4 weak features
train = train.drop(columns=drop_cols)
test  = test.drop(columns=drop_cols)

print(f"\nFinal train shape: {train.shape}")
print(f"Final test shape:  {test.shape}")
print(f"\nRemaining features: {[c for c in train.columns if c not in [target, 'Log Target Pressure']]}")


Final train shape: (9990, 21)
Final test shape:  (3203, 19)

Remaining features: ['Tank Failure Pressure (bar)', 'Liquid Ratio (%)', 'Tank Width (m)', 'Tank Length (m)', 'Tank Height (m)', 'BLEVE Height (m)', 'Vapour Height (m)', 'Vapour Temperature (K)', 'Liquid Temperature (K)', 'Obstacle Distance to BLEVE (m)', 'Obstacle Width (m)', 'Obstacle Height (m)', 'Obstacle Thickness (m)', 'Obstacle Angle', 'Sensor ID', 'Sensor Position Side', 'Sensor Position x', 'Sensor Position y', 'Sensor Position z']


## **Feature Engineering:**
You are encouraged to derive new features that could improve
model performance. For instance, the ratio Tank Width
Tank Length may serve as a useful additional
feature.

## **Data Type Conversion:**
Convert features to appropriate data types as required by
your model. For example, categorical variables may need to be encoded into numerical
formats using one-hot encoding or similar techniques

In [ ]:
#Engineer new features
#Tank Volume (m³)
for df in [train, test]:
    df['Tank Volume'] = (df['Tank Width (m)']
                         * df['Tank Length (m)']
                         * df['Tank Height (m)'])


In [ ]:
#Sensor Distance to BLEVE (m)
for df in [train, test]:
    df['Sensor Distance to BLEVE'] = np.sqrt(
        df['Sensor Position x']**2 +
        df['Sensor Position y']**2 +
        df['Sensor Position z']**2
    )

In [ ]:
#Obstacle Volume (m³
for df in [train, test]:
    df['Obstacle Volume'] = (df['Obstacle Width (m)']
                              * df['Obstacle Height (m)']
                              * df['Obstacle Thickness (m)'])

In [ ]:
#Obstacle Face Area (m²)
for df in [train, test]:
    df['Obstacle Face Area'] = (df['Obstacle Width (m)']
                                 * df['Obstacle Height (m)'])

In [ ]:
#Verify improvement with cross-validation
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score

target = 'Target Pressure (bar)'

X = train.drop(columns=[target, 'Log Target Pressure'])
y = train['Log Target Pressure']

rf = RandomForestRegressor(n_estimators=100, random_state=0, n_jobs=-1)
r2 = cross_val_score(rf, X, y, cv=5, scoring='r2').mean()

print(f"R² after feature engineering: {r2:.4f}")
# Expect ~0.8876 vs 0.8735 baseline — improvement of +0.014

# --- Cell 22: Final feature summary ---
feature_cols = [c for c in train.columns
                if c not in [target, 'Log Target Pressure']]

print(f"\nTotal features going into modelling: {len(feature_cols)}")
print("Features:")
for f in feature_cols:
    tag = ' ** engineered **' if f in ['Tank Volume', 'Sensor Distance to BLEVE',
                                        'Obstacle Volume', 'Obstacle Face Area'] else ''
    print(f"  {f}{tag}")

R² after feature engineering: 0.8879

Total features going into modelling: 23
Features:
  Tank Failure Pressure (bar)
  Liquid Ratio (%)
  Tank Width (m)
  Tank Length (m)
  Tank Height (m)
  BLEVE Height (m)
  Vapour Height (m)
  Vapour Temperature (K)
  Liquid Temperature (K)
  Obstacle Distance to BLEVE (m)
  Obstacle Width (m)
  Obstacle Height (m)
  Obstacle Thickness (m)
  Obstacle Angle
  Sensor ID
  Sensor Position Side
  Sensor Position x
  Sensor Position y
  Sensor Position z
  Tank Volume ** engineered **
  Sensor Distance to BLEVE ** engineered **
  Obstacle Volume ** engineered **
  Obstacle Face Area ** engineered **


## **Feature Scaling:**

Apply normalization or standardization where appropriate, particularly for models sensitive to input magnitudes.

In [ ]:
# Prepare X and y
target = 'Target Pressure (bar)'

feature_cols = [c for c in train.columns
                if c not in [target, 'Log Target Pressure']]

X = train[feature_cols].copy()
y = train['Log Target Pressure'].copy()   # log-transformed target for all models

X_test = test[feature_cols].copy()

print(f"X shape:      {X.shape}")
print(f"X_test shape: {X_test.shape}")

X shape:      (9990, 23)
X_test shape: (3203, 23)


In [ ]:
# Apply StandardScaler
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_scaled      = scaler.fit_transform(X)       # fit + transform train
X_test_scaled = scaler.transform(X_test)      # transform test only

# Convert back to DataFrames to keep column names accessible
X_scaled      = pd.DataFrame(X_scaled,      columns=feature_cols)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=feature_cols)

In [ ]:
# Verify scaling
print("=== Scaled feature stats (should be ~mean=0, std=1) ===")
print(X_scaled.describe().loc[['mean', 'std']].round(3))

=== Scaled feature stats (should be ~mean=0, std=1) ===
      Tank Failure Pressure (bar)  Liquid Ratio (%)  Tank Width (m)  \
mean                          0.0              -0.0            -0.0   
std                           1.0               1.0             1.0   

      Tank Length (m)  Tank Height (m)  BLEVE Height (m)  Vapour Height (m)  \
mean              0.0              0.0              -0.0               -0.0   
std               1.0              1.0               1.0                1.0   

      Vapour Temperature (K)  Liquid Temperature (K)  \
mean                     0.0                    -0.0   
std                      1.0                     1.0   

      Obstacle Distance to BLEVE (m)  ...  Obstacle Angle  Sensor ID  \
mean                            -0.0  ...             0.0        0.0   
std                              1.0  ...             1.0        1.0   

      Sensor Position Side  Sensor Position x  Sensor Position y  \
mean                  -0.0            

In [ ]:
# Summary of what we have going into modelling
print("\n=== Ready for Modelling ===")
print(f"X          — unscaled, shape {X.shape}         → use for Random Forest / XGBoost")
print(f"X_scaled   — standardised, shape {X_scaled.shape}  → use for Linear Regression / Neural Network")
print(f"y          — log(Target Pressure), shape {y.shape}")
print(f"X_test     — unscaled test set")
print(f"X_test_scaled — scaled test set")


=== Ready for Modelling ===
X          — unscaled, shape (9990, 23)         → use for Random Forest / XGBoost
X_scaled   — standardised, shape (9990, 23)  → use for Linear Regression / Neural Network
y          — log(Target Pressure), shape (9990,)
X_test     — unscaled test set
X_test_scaled — scaled test set


# **Model Development**

In this phase of the assignment, your objective is to explore and compare a diverse set of
machine learning models to determine the most effective approach for predicting peak pressure in BLEVE scenarios. A key requirement is to evaluate at least three fundamentally
different types of machine learning models. These models must be assessed using at
least two different performance metrics.


## **Model Selection:**

You must examine at least three distinct types of machine learning
models. Examples include linear regression models, support vector regression (SVR),
decision tree-based models (e.g., Random Forest, XGBoost), and neural networks.
Note that models of a similar nature—such as Random Forest and Gradient Boosted
Decision Trees (GBDT)—are not considered sufficiently distinct and will not fulfil the
requirement on their own.


In [ ]:
# Define metrics helper
from sklearn.model_selection import KFold, cross_val_score
from sklearn.metrics import r2_score, mean_absolute_percentage_error, root_mean_squared_error
import warnings
warnings.filterwarnings('ignore')

def evaluate_model(model, X, y, cv=5):
    """
    Returns mean CV scores: R2, MAPE (on original scale), RMSE (log scale).
    Predictions are made in log space then exp()-converted for MAPE.
    """
    kf = KFold(n_splits=cv, shuffle=True, random_state=0)
    r2_scores, mape_scores, rmse_scores = [], [], []

    for train_idx, val_idx in kf.split(X):
        X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]

        model.fit(X_tr, y_tr)
        log_preds = model.predict(X_val)

        # Back-transform for MAPE (original bar scale)
        preds_bar    = np.exp(log_preds)
        y_val_bar    = np.exp(y_val)

        r2_scores.append(r2_score(y_val, log_preds))
        mape_scores.append(mean_absolute_percentage_error(y_val_bar, preds_bar))
        rmse_scores.append(root_mean_squared_error(y_val, log_preds))

    return {
        'R2':   np.mean(r2_scores).round(4),
        'MAPE': np.mean(mape_scores).round(4),
        'RMSE': np.mean(rmse_scores).round(4),
    }

In [ ]:
# Model 1 — Linear Regression
from sklearn.linear_model import LinearRegression

lr = LinearRegression()
lr_scores = evaluate_model(lr, X_scaled, y)
print("Model 1 — Linear Regression (baseline)")
print(f"  R²:   {lr_scores['R2']}")
print(f"  MAPE: {lr_scores['MAPE']}")
print(f"  RMSE: {lr_scores['RMSE']}")
print()

# Train R2 to check underfitting
lr.fit(X_scaled, y)
print(f"  Train R² (full set): {lr.score(X_scaled, y):.4f}")
print("  --> Train R2 ≈ CV R2 → model is underfitting, not overfitting.")
print("      The relationship is genuinely non-linear. Linear Regression")
print("      is retained as a baseline but will not be the final model.")

Model 1 — Linear Regression (baseline)
  R²:   0.7679
  MAPE: 0.4521
  RMSE: 0.4771

  Train R² (full set): 0.7692
  --> Train R2 ≈ CV R2 → model is underfitting, not overfitting.
      The relationship is genuinely non-linear. Linear Regression
      is retained as a baseline but will not be the final model.


In [ ]:
# Model 2 — Random Forest
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(n_estimators=100, random_state=0, n_jobs=-1)
rf_scores = evaluate_model(rf, X, y)
print("\nModel 2 — Random Forest (default params)")
print(f"  R²:   {rf_scores['R2']}")
print(f"  MAPE: {rf_scores['MAPE']}")
print(f"  RMSE: {rf_scores['RMSE']}")


Model 2 — Random Forest (default params)
  R²:   0.9557
  MAPE: 0.1622
  RMSE: 0.2082


In [ ]:
# Model 3 — Neural Network (MLP)
from sklearn.neural_network import MLPRegressor

mlp = MLPRegressor(
    hidden_layer_sizes=(100,),   # 1 hidden layer, 100 neurons — default starting point
    max_iter=500,
    random_state=0
)
mlp_scores = evaluate_model(mlp, X_scaled, y)
print("\nModel 3 — Neural Network / MLP (default params)")
print(f"  R²:   {mlp_scores['R2']}")
print(f"  MAPE: {mlp_scores['MAPE']}")
print(f"  RMSE: {mlp_scores['RMSE']}")


Model 3 — Neural Network / MLP (default params)
  R²:   0.9652
  MAPE: 0.1437
  RMSE: 0.1847


In [ ]:
# Comparison table (default params)

print("\n=== Model Comparison — Default Hyperparameters (5-fold CV) ===")
print(f"{'Model':<25} {'R²':>8} {'MAPE':>8} {'RMSE':>8}")
print("-" * 52)
results = {
    'Linear Regression':  lr_scores,
    'Random Forest':      rf_scores,
    'Neural Network':     mlp_scores,
}
for name, scores in results.items():
    print(f"{name:<25} {scores['R2']:>8} {scores['MAPE']:>8} {scores['RMSE']:>8}")

print()
print("Observations:")
print("  - Linear Regression underfits (train R2 ≈ CV R2 ≈ 0.73) — the")
print("    pressure-feature relationship is non-linear.")
print("  - Random Forest and Neural Network both capture non-linearity well.")
print("  - Neural Network leads on all metrics at default settings.")
print("  - All 3 models are fundamentally different in architecture.")
print("  - Hyperparameter tuning in the next step should improve RF and MLP further.")


=== Model Comparison — Default Hyperparameters (5-fold CV) ===
Model                           R²     MAPE     RMSE
----------------------------------------------------
Linear Regression           0.7679   0.4521   0.4771
Random Forest               0.9557   0.1622   0.2082
Neural Network              0.9652   0.1437   0.1847

Observations:
  - Linear Regression underfits (train R2 ≈ CV R2 ≈ 0.73) — the
    pressure-feature relationship is non-linear.
  - Random Forest and Neural Network both capture non-linearity well.
  - Neural Network leads on all metrics at default settings.
  - All 3 models are fundamentally different in architecture.
  - Hyperparameter tuning in the next step should improve RF and MLP further.


## **Hyperparameter Tuning:**

Each model has associated hyperparameters that significantly influence its performance. You are expected to tune these hyperparameters
using appropriate strategies such as hold-out validation or cross-validation. Techniques
such as grid search or random search may be used to identify optimal parameter values.

In [ ]:
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, KFold
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import r2_score, mean_absolute_percentage_error
import warnings
warnings.filterwarnings('ignore')

kf = KFold(n_splits=5, shuffle=True, random_state=0)

In [ ]:
## Ridge Regression
# Tune Ridge
ridge_param_grid = {
    'alpha': [0.01, 0.1, 1, 10, 100, 1000]
}

ridge_search = GridSearchCV(
    Ridge(),
    ridge_param_grid,
    cv=kf,
    scoring='r2',
    n_jobs=-1
)
ridge_search.fit(X_scaled, y)

best_ridge = ridge_search.best_estimator_
print("── Ridge Regression ──")
print(f"  Best alpha:  {ridge_search.best_params_['alpha']}")
print(f"  Best CV R²:  {ridge_search.best_score_:.4f}")

# Show full grid results
print("\n  Full grid results:")
ridge_results = pd.DataFrame(ridge_search.cv_results_)
for _, row in ridge_results[['param_alpha','mean_test_score']].iterrows():
    print(f"    alpha={row['param_alpha']:<8}  R²={row['mean_test_score']:.4f}")

── Ridge Regression ──
  Best alpha:  0.01
  Best CV R²:  0.7679

  Full grid results:
    alpha=0.01      R²=0.7679
    alpha=0.1       R²=0.7679
    alpha=1.0       R²=0.7679
    alpha=10.0      R²=0.7671
    alpha=100.0     R²=0.7413
    alpha=1000.0    R²=0.5970


In [ ]:
## Random Forest
# Tune Random Forest

rf_param_dist = {
    'n_estimators':      [100, 200, 300],
    'max_depth':         [None, 10, 20, 30],
    'min_samples_leaf':  [1, 2, 4],
    'min_samples_split': [2, 5, 10],
    'max_features':      ['sqrt', 'log2', 0.5],
}

rf_search = RandomizedSearchCV(
    RandomForestRegressor(random_state=0, n_jobs=-1),
    rf_param_dist,
    n_iter=30,           # try 30 random combinations
    cv=kf,
    scoring='r2',
    random_state=0,
    n_jobs=-1
)
rf_search.fit(X, y)    # unscaled — trees are scale-invariant

best_rf = rf_search.best_estimator_
print("\n── Random Forest ──")
print(f"  Best params: {rf_search.best_params_}")
print(f"  Best CV R²:  {rf_search.best_score_:.4f}")


── Random Forest ──
  Best params: {'n_estimators': 300, 'min_samples_split': 2, 'min_samples_leaf': 2, 'max_features': 0.5, 'max_depth': None}
  Best CV R²:  0.9571


In [ ]:
## Neural Network (MLP)
# Tune Neural Network

mlp_param_dist = {
    'hidden_layer_sizes': [(100,), (200,), (100, 50), (200, 100), (100, 50, 25)],
    'alpha':              [0.0001, 0.001, 0.01, 0.1],
    'learning_rate_init': [0.001, 0.005, 0.01],
}

mlp_search = RandomizedSearchCV(
    MLPRegressor(max_iter=500, random_state=0),
    mlp_param_dist,
    n_iter=20,           # 20 random combinations
    cv=kf,
    scoring='r2',
    random_state=123,
    n_jobs=-1
)
mlp_search.fit(X_scaled, y)   # scaled — MLP requires it

best_mlp = mlp_search.best_estimator_
print("\n── Neural Network (MLP) ──")
print(f"  Best params: {mlp_search.best_params_}")
print(f"  Best CV R²:  {mlp_search.best_score_:.4f}")


── Neural Network (MLP) ──
  Best params: {'learning_rate_init': 0.005, 'hidden_layer_sizes': (200, 100), 'alpha': 0.01}
  Best CV R²:  0.9778


In [ ]:
# Evaluate tuned models with MAPE + R²
ridge_tuned = evaluate_model(best_ridge, X_scaled, y)
rf_tuned    = evaluate_model(best_rf,    X,        y)
mlp_tuned   = evaluate_model(best_mlp,  X_scaled, y)

print("\n=== Tuned Model Comparison (5-fold CV) ===")
print(f"{'Model':<25} {'R²':>8} {'MAPE':>8} {'RMSE':>8}")
print("─" * 52)
tuned = {
    'Ridge Regression': ridge_tuned,
    'Random Forest':    rf_tuned,
    'Neural Network':   mlp_tuned,
}
for name, scores in tuned.items():
    print(f"{name:<25} {scores['R2']:>8} {scores['MAPE']:>8} {scores['RMSE']:>8}")



=== Tuned Model Comparison (5-fold CV) ===
Model                           R²     MAPE     RMSE
────────────────────────────────────────────────────
Ridge Regression            0.7679   0.4521   0.4771
Random Forest               0.9571   0.1603   0.2051
Neural Network              0.9778   0.1129   0.1475


In [ ]:
# Select final model
best_model_name = min(tuned, key=lambda k: tuned[k]['MAPE'])
print(f"\nBest model by MAPE: {best_model_name}")
print("This model will be used for final test set predictions.")

# Map name to model + correct X version
model_map = {
    'Ridge Regression': (best_ridge, X_scaled, X_test_scaled),
    'Random Forest':    (best_rf,    X,        X_test),
    'Neural Network':   (best_mlp,  X_scaled, X_test_scaled),
}
final_model, X_final_train, X_final_test = model_map[best_model_name]


Best model by MAPE: Neural Network
This model will be used for final test set predictions.


## **Evaluation Metrics:**

All models must be evaluated using at least two metrics. The
use of Mean Absolute Percentage Error (MAPE) and R2
is compulsory, both of which are available in the Scikit-learn library. Additional metrics such as Root Mean
Squared Error (RMSE) or Mean Absolute Error (MAE) may be included for a more
comprehensive analysis.


In [ ]:
from sklearn.metrics import (r2_score, mean_absolute_percentage_error,
                              mean_squared_error, mean_absolute_error)
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Hold-out validation split

target = 'Target Pressure (bar)'
feature_cols = [c for c in train.columns if c not in [target, 'Log Target Pressure']]

X_all = train[feature_cols].copy()
y_all = train['Log Target Pressure'].copy()

X_tr, X_val, y_tr, y_val = train_test_split(
    X_all, y_all, test_size=0.2, random_state=0
)

from sklearn.preprocessing import StandardScaler
val_scaler   = StandardScaler()
X_tr_sc      = pd.DataFrame(val_scaler.fit_transform(X_tr),  columns=feature_cols)
X_val_sc     = pd.DataFrame(val_scaler.transform(X_val),     columns=feature_cols)

print(f"Train split:      {X_tr.shape[0]} rows")
print(f"Validation split: {X_val.shape[0]} rows")


Train split:      7992 rows
Validation split: 1998 rows


In [ ]:
# Metrics helper

def compute_metrics(y_true_log, y_pred_log):
    """
    Computes R², MAPE, RMSE, MAE.
    MAPE and MAE are on original bar scale (exp-transformed).
    R² and RMSE are on log scale (consistent with model training).
    """
    y_true_bar = np.exp(y_true_log)
    y_pred_bar = np.exp(y_pred_log)
    return {
        'R²':   round(r2_score(y_true_log,  y_pred_log),                       4),
        'MAPE': round(mean_absolute_percentage_error(y_true_bar, y_pred_bar),   4),
        'RMSE': round(np.sqrt(mean_squared_error(y_true_log, y_pred_log)),      4),
        'MAE':  round(mean_absolute_error(y_true_bar, y_pred_bar),              4),
    }

In [ ]:
# Fit and evaluate all 3 tuned models

results = {}

# Ridge
best_ridge.fit(X_tr_sc, y_tr)
results['Ridge Regression'] = {
    'Train': compute_metrics(y_tr.values,  best_ridge.predict(X_tr_sc)),
    'Val':   compute_metrics(y_val.values, best_ridge.predict(X_val_sc)),
}

# Random Forest
best_rf.fit(X_tr, y_tr)
results['Random Forest'] = {
    'Train': compute_metrics(y_tr.values,  best_rf.predict(X_tr)),
    'Val':   compute_metrics(y_val.values, best_rf.predict(X_val)),
}

# Neural Network
best_mlp.fit(X_tr_sc, y_tr)
results['Neural Network'] = {
    'Train': compute_metrics(y_tr.values,  best_mlp.predict(X_tr_sc)),
    'Val':   compute_metrics(y_val.values, best_mlp.predict(X_val_sc)),
}

In [ ]:
# Print results table
print("\n=== Model Evaluation — Train vs Validation ===")
print(f"{'Model':<22} {'Set':<6} {'R²':>7} {'MAPE':>7} {'RMSE':>7} {'MAE':>7}")
print("─" * 56)

for model_name, sets in results.items():
    for set_name, m in sets.items():
        print(f"{model_name:<22} {set_name:<6} "
              f"{m['R²']:>7} {m['MAPE']:>7} {m['RMSE']:>7} {m['MAE']:>7}")
    print()


=== Model Evaluation — Train vs Validation ===
Model                  Set         R²    MAPE    RMSE     MAE
────────────────────────────────────────────────────────
Ridge Regression       Train   0.7704  0.4505  0.4786  0.1509
Ridge Regression       Val      0.763  0.4351  0.4668  0.1362

Random Forest          Train   0.9908  0.0723  0.0957  0.0285
Random Forest          Val     0.9571  0.1567  0.1986  0.0538

Neural Network         Train   0.9892  0.0784  0.1037  0.0256
Neural Network         Val     0.9772  0.1078  0.1447  0.0338



In [ ]:
# Interpret the results
print("=== Observations ===")
print()
print("Ridge Regression:")
print("  Train R² ≈ Val R² ≈ 0.77 — consistent underfitting.")
print("  MAPE ~0.41 on validation. The model cannot capture")
print("  non-linear blast pressure decay. Retained as baseline only.")
print()
print("Random Forest:")
print("  Train R² = 0.99, Val R² = 0.96 — some overfitting present")
print("  but validation performance is still strong.")
print("  Val MAPE ≈ 0.16, well within the top scoring bracket (<0.20).")
print()
print("Neural Network:")
print("  Train R² = 0.99, Val R² = 0.97 — best generalisation.")
print("  Val MAPE ≈ 0.13 — lowest of all three models.")
print("  Selected as the final model for Kaggle submission.")
print()
print("Note: Final Kaggle (test) MAPE will be reported after submission.")
print("      It may differ slightly from validation MAPE due to")
print("      natural variation between the validation and test distributions.")

=== Observations ===

Ridge Regression:
  Train R² ≈ Val R² ≈ 0.77 — consistent underfitting.
  MAPE ~0.41 on validation. The model cannot capture
  non-linear blast pressure decay. Retained as baseline only.

Random Forest:
  Train R² = 0.99, Val R² = 0.96 — some overfitting present
  but validation performance is still strong.
  Val MAPE ≈ 0.16, well within the top scoring bracket (<0.20).

Neural Network:
  Train R² = 0.99, Val R² = 0.97 — best generalisation.
  Val MAPE ≈ 0.13 — lowest of all three models.
  Selected as the final model for Kaggle submission.

Note: Final Kaggle (test) MAPE will be reported after submission.
      It may differ slightly from validation MAPE due to
      natural variation between the validation and test distributions.


## **Submission**

In [ ]:
#Save test IDs BEFORE dropping the ID column
import pandas as pd
import numpy as np

test_ids = pd.read_csv('/content/drive/MyDrive/COMP3010/test.csv')['ID'].copy()
print(f"Test IDs: {test_ids.min()} to {test_ids.max()}, count: {len(test_ids)}")  # 0–3202, 3203 rows

Test IDs: 0 to 3202, count: 3203


In [ ]:
#Fit final model on ALL training data
final_model.fit(X_final_train, y_all)   # y_all = full log-transformed target
print(f"Final model retrained on {X_final_train.shape[0]} rows (full training set)")

Final model retrained on 9990 rows (full training set)


In [ ]:
#Generate predictions
log_preds = final_model.predict(X_final_test)
predictions_bar = np.exp(log_preds)

print(f"\nPrediction stats:")
print(f"  Min:  {predictions_bar.min():.4f} bar")
print(f"  Max:  {predictions_bar.max():.4f} bar")
print(f"  Mean: {predictions_bar.mean():.4f} bar")
print(f"  Std:  {predictions_bar.std():.4f} bar")

# Sanity check — should look similar to training target distribution
print(f"\nTraining target stats for comparison:")
print(f"  Min:  {np.exp(y_all).min():.4f} bar")
print(f"  Max:  {np.exp(y_all).max():.4f} bar")
print(f"  Mean: {np.exp(y_all).mean():.4f} bar")



Prediction stats:
  Min:  0.0237 bar
  Max:  5.2501 bar
  Mean: 0.3201 bar
  Std:  0.3925 bar

Training target stats for comparison:
  Min:  0.0161 bar
  Max:  9.1705 bar
  Mean: 0.3602 bar


In [ ]:
#Build submission DataFrame
submission = pd.DataFrame({
    'ID':                    test_ids.values,
    'Target Pressure (bar)': predictions_bar,
})

print(f"\nSubmission shape: {submission.shape}")   # must be (3203, 2)
print(submission.head(10))


Submission shape: (3203, 2)
   ID  Target Pressure (bar)
0   0               0.237938
1   1               0.261865
2   2               0.365217
3   3               0.327140
4   4               0.314813
5   5               0.277504
6   6               0.151498
7   7               0.189860
8   8               0.322207
9   9               1.638338


In [ ]:
#Save to Google Drive and download
output_path = '/content/drive/MyDrive/COMP3010/prediction.csv'
submission.to_csv(output_path, index=False)
print(f"\nSaved to: {output_path}")


from google.colab import files
submission.to_csv('prediction.csv', index=False)
files.download('prediction.csv')

print("\nDone! Upload prediction.csv to Kaggle.")


Saved to: /content/drive/MyDrive/COMP3010/prediction.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


Done! Upload prediction.csv to Kaggle.
